# PathoVision AI: training on Google Colab (free T4 GPU)

**Before running:** go to **Runtime → Change runtime type → T4 GPU → Save**, then **Runtime → Run all**.

- When asked, allow **Google Drive** access. Models and results are saved to `MyDrive/pathovision/`, so a Colab disconnect does not lose finished work.
- If Colab disconnects, reconnect and **Run all** again. Finished steps are detected in Drive and skipped.
- Keep the tab open, and click inside the page every ~30-60 min (free Colab stops idle sessions).

| Step | What | Time (T4, approx.) |
|---|---|---|
| 1 | Setup + Drive | 2 min |
| 2-3 | PCam + ResNet-18 classifier (**skipped if already trained**) | 0 or 30-50 min |
| 4 | Download CAMELYON16 slides (18 tumour + 6 normal by default, ~35-45 GB) | 25-35 min |
| 5 | Extract patches, train U-Net (30 epochs) | 40-50 min |
| 6 | Whole-slide inference + lesion-level evaluation on 5 held-out slides | 10-15 min |

Change `RUN` in the config cell to start a fresh segmentation run without overwriting earlier ones.

## 1. Setup

In [ ]:
!rm -rf /content/pathovision-ai
import torch
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then run again."
print(torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, glob, json
OUT = "/content/drive/MyDrive/pathovision"      # persistent: models, metrics, reports
DATA = "/content/data"                           # scratch: datasets (re-downloaded after a disconnect)
os.makedirs(OUT, exist_ok=True); os.makedirs(DATA, exist_ok=True)
os.environ["DATA"] = DATA

# ---- run configuration ----
RUN = "v2"                                  # results: runs/seg_resnet34_<RUN>, outputs/holdout_<RUN>
N_TUMOR, N_NORMAL = 18, 6                   # CAMELYON16 slides to download
N_HOLDOUT_TUMOR, N_HOLDOUT_NORMAL = 3, 2    # never seen in training: final evaluation
N_VAL_TUMOR, N_VAL_NORMAL = 2, 1            # used to pick the best epoch
SEG_EPOCHS = 30

In [ ]:
# Get the code: clone from GitHub, or upload pathovision-ai.zip if the repo isn't pushed yet
import subprocess
REPO_URL = "https://github.com/nirdoshk0228/pathovision-ai.git"
CODE = "/content/pathovision-ai"
if not os.path.exists(f"{CODE}/scripts"):
    subprocess.run(["rm", "-rf", CODE])
    subprocess.run(["git", "clone", "-q", REPO_URL, CODE])
    if not os.path.exists(f"{CODE}/scripts"):
        subprocess.run(["rm", "-rf", CODE])
        from google.colab import files
        print("Code not found on GitHub (repo empty or not pushed). Upload pathovision-ai.zip:")
        files.upload()
        !unzip -q -o pathovision-ai.zip -d /content
assert os.path.exists(f"{CODE}/pathovision/data/augment.py"), "code incomplete: push the latest repo or upload the zip"
%cd /content/pathovision-ai
!git log --oneline | head -3
!pip install -q openslide-python openslide-bin imagecodecs
!df -h /content | tail -1

## 2. PatchCamelyon (327,680 patches, 96x96 px @ 10x, slide-disjoint splits)

In [ ]:
CLS_DIR = f"{OUT}/runs/cls_resnet18"
if not os.path.exists(f"{CLS_DIR}/metrics.json"):
    !bash scripts/download_data.sh pcam
else:
    print("classifier already trained, skipping PCam download")

## 3. Patch classifier

ImageNet-pretrained ResNet-18, AdamW + one-cycle LR, mixed precision, HED stain + D4 augmentation.
The threshold is chosen on **validation**, and the final numbers are on the untouched **test** split with 8-view TTA.

In [ ]:
if not os.path.exists(f"{CLS_DIR}/metrics.json"):
    !python -m pathovision.training.train_classifier --data-format pcam_h5 --data-dir {DATA}/pcam \
        --arch resnet18 --epochs 5 --batch-size 256 --lr 1e-3 --workers 2 --tta 8 --out-dir {CLS_DIR}
    !rm -rf {DATA}/pcam

In [ ]:
from IPython.display import Image, display
m = json.load(open(f"{CLS_DIR}/metrics.json"))["results"]
for split, r in m.items():
    print(f"{split:9s} AUC {r['roc_auc']:.4f}  acc {r['accuracy']:.4f}  precision {r['precision']:.4f}  recall {r['recall']:.4f}  F1 {r['f1']:.4f}")
display(Image(f"{CLS_DIR}/roc_test.png"))

## 4. CAMELYON16 whole-slide images

From the AWS Open Data bucket `s3://camelyon-dataset` (no account needed). Each slide is 0.5-3 GB.

In [ ]:
!N_TUMOR={N_TUMOR} N_NORMAL={N_NORMAL} bash scripts/download_data.sh camelyon
!du -sh {DATA}/camelyon16 && df -h /content | tail -1

## 5. Patch extraction + U-Net segmentation

Splits are **by slide**: validation slides pick the best epoch; held-out slides are only used in step 6.
Normal slides add hard negatives so the U-Net learns what is *not* tumour.

In [ ]:
imgs = sorted(glob.glob(f"{DATA}/camelyon16/images/*.tif")) + sorted(glob.glob(f"{DATA}/holdout/*.tif"))
names = sorted({os.path.basename(p)[:-4] for p in imgs})
tumour = [s for s in names if s.startswith("tumor")]
normal = [s for s in names if s.startswith("normal")]
HOLDOUT = tumour[-N_HOLDOUT_TUMOR:] + normal[-N_HOLDOUT_NORMAL:]
VAL = tumour[-(N_HOLDOUT_TUMOR + N_VAL_TUMOR):-N_HOLDOUT_TUMOR] + normal[-(N_HOLDOUT_NORMAL + N_VAL_NORMAL):-N_HOLDOUT_NORMAL]
VAL_ARG = " ".join(VAL)
os.makedirs(f"{DATA}/holdout", exist_ok=True)
for s in HOLDOUT:
    src = f"{DATA}/camelyon16/images/{s}.tif"
    if os.path.exists(src):
        os.replace(src, f"{DATA}/holdout/{s}.tif")
print("training slides:  ", [s for s in names if s not in HOLDOUT and s not in VAL])
print("validation slides:", VAL)
print("held out:         ", HOLDOUT)

In [ ]:
SEG_DIR = f"{OUT}/runs/seg_resnet34_{RUN}"
if not os.path.exists(f"{SEG_DIR}/metrics.json"):
    !rm -rf {DATA}/patches
    !python scripts/extract_patches.py --slides "{DATA}/camelyon16/images/*.tif" \
        --xml-dir {DATA}/camelyon16/annotations --out {DATA}/patches --val-slides {VAL_ARG} --test-slides none \
        --no-cls --seg-size 256 --seg-mpp 0.972 --seg-per-slide 800 --seg-per-normal-slide 300 --seg-neg-frac 0.35
print("train patches:", len(os.listdir(f"{DATA}/patches/seg/train/images")) if os.path.isdir(f"{DATA}/patches/seg/train/images") else "skipped",
      "| val patches:", len(os.listdir(f"{DATA}/patches/seg/valid/images")) if os.path.isdir(f"{DATA}/patches/seg/valid/images") else "skipped")

In [ ]:
if not os.path.exists(f"{SEG_DIR}/metrics.json"):
    !python -m pathovision.training.train_segmentation --data-dir {DATA}/patches/seg \
        --arch resnet34_unet --epochs {SEG_EPOCHS} --patience 8 --batch-size 16 --lr 3e-4 --stain-sigma 0.08 \
        --workers 2 --tta 8 --out-dir {SEG_DIR}
sm = json.load(open(f"{SEG_DIR}/metrics.json"))
print(sm["results"])
for h in sm["history"]:
    print("  epoch", h["epoch"], "loss", round(h["train_loss"], 4), "val_dice", round(h["val_dice"], 4))

## 6. Whole-slide inference on the held-out slides

The U-Net output is kept only where the patch classifier agrees (cascade), which removes most isolated false positives.
Reported: pixel Dice on tumour slides, lesion-level sensitivity, and false-positive lesions per slide.

In [ ]:
!mkdir -p {OUT}/models
!cp {CLS_DIR}/best.pt {OUT}/models/classifier.pt
!cp {SEG_DIR}/best.pt {OUT}/models/segmenter_{RUN}.pt
EVAL_DIR = f"{OUT}/outputs/holdout_{RUN}"
!python scripts/evaluate_slides.py --slides "{DATA}/holdout/*.tif" --xml-dir {DATA}/camelyon16/annotations \
    --cls-ckpt {OUT}/models/classifier.pt --seg-ckpt {OUT}/models/segmenter_{RUN}.pt \
    --out {EVAL_DIR} --workers 2

In [ ]:
for d in sorted(glob.glob(f"{EVAL_DIR}/*/")):
    r = json.load(open(d + "report.json"))
    q, b = r["quantification"], r["benchmark"]
    print(f"\n{os.path.basename(d[:-1])}: {r['slide']['width']:,} x {r['slide']['height']:,} px | "
          f"{b['tiles_classified']:,} patches ({b['tiles_skipped_background_pct']}% background skipped) | "
          f"{b['classification_tiles_per_sec']} patches/s | total {b['total_seconds']} s | "
          f"peak RAM {b['peak_rss_mb']} MB, GPU {b['peak_gpu_mb']} MB | tissue {q['tissue_area_mm2']:.1f} mm2")
    print(f"  lesions {q['n_lesions']} | largest {q['largest_lesion_diameter_mm']:.2f} mm | {q['slide_category']} | "
          f"tumour {q['tumor_area_mm2']:.3f} mm2 | GT {r.get('ground_truth')}")
    display(Image(d + "tissue_mask.png", width=450))
    display(Image(d + "heatmap_overlay.png", width=700))
    display(Image(d + "prediction_overlay.png", width=700))
print(open(f"{EVAL_DIR}/summary.json").read())
print(open(f"{EVAL_DIR}/per_slide.csv").read())

In [ ]:
# Throughput vs batch size (for the README performance table)
bench_slide = sorted(glob.glob(f"{DATA}/holdout/tumor_*.tif"))[0]
!python scripts/benchmark.py --slide {bench_slide} --cls-ckpt {OUT}/models/classifier.pt --batch-sizes 64 256 512 --workers 2

## Done

Everything is in Google Drive → `MyDrive/pathovision/`:
- `models/classifier.pt`, `models/segmenter_<RUN>.pt`
- `runs/*/metrics.json`, `runs/cls_resnet18/roc_test.png`: the numbers for README and resume
- `outputs/holdout_<RUN>/summary.json` and `per_slide.csv`: Dice, lesion sensitivity, false positives per slide
- `outputs/holdout_<RUN>/<slide>/heatmap_overlay.png` and `prediction_overlay.png`: screenshots for `assets/`